# NLP Workshop â€” Notebook 08 (Claude Assisted)
# Retrieval-Augmented Generation (RAG) and LLM-based Retrieval

**Series:** Classical NLP to Modern Embeddings  
**Dataset:** Quran Translations (sahih column, 6,236 verses)  
**Prerequisites:** NB06 (Sentence Embeddings), NB07 (FAISS)

In this final notebook we connect everything: retrieve relevant verses with FAISS,
inject them into a prompt, and pass the prompt to a small language model.

## What is RAG?

Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a generative language model. Instead of relying solely on knowledge baked into model weights, the model is given relevant passages at inference time.

There are three stages:

- **Retrieve** â€” semantic search returns top-k passages most relevant to the query
- **Augment** â€” those passages are inserted into the prompt as context, grounding the model
- **Generate** â€” an LLM reads the grounded prompt and produces a factual, cited answer

### Pipeline

```
Query
  |
  v
[Encoder] --> query vector
  |
  v
[FAISS Index] --> top-k verses
  |
  v
[Prompt Builder] --> "Context: ...\n\nQuestion: ..."
  |
  v
[LLM (flan-t5-small)] --> Answer
```

### Why RAG beats "just ask the LLM"

| Property | Bare LLM | RAG |
|---|---|---|
| Knowledge source | Parametric (weights) | Retrieved documents |
| Hallucination risk | High | Reduced (grounded) |
| Citability | None | Verse IDs cited |
| Updatable | Requires retraining | Swap the index |

RAG anchors every answer to real passages, makes hallucinations easier to detect, and lets you update the knowledge base without touching the model.

## Configuration

In [1]:
import re, time, warnings
import numpy as np
import pandas as pd
import faiss
import torch
import nltk
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

warnings.filterwarnings("ignore")

# --- auto-download NLTK data ---
for _pkg, _res in [("punkt", "tokenizers/punkt"),
                   ("punkt_tab", "tokenizers/punkt_tab"),
                   ("stopwords", "corpora/stopwords")]:
    try:
        nltk.data.find(_res)
    except LookupError:
        nltk.download(_pkg, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# --- Config ---
MODEL_NAME   = "all-MiniLM-L6-v2"
GEN_MODEL    = "google/flan-t5-small"
USE_LOCAL_LLM = True          # set False to skip generation (retrieval only)
TEXT_COL       = "sahih"         # column used for embeddings
EMBEDDINGS_DIR = Path("..") / "embeddings"  # shared cache folder
TOP_K        = 4              # passages to retrieve
MAX_NEW_TOKENS = 96
NUM_BEAMS    = 2
CONTEXT_MAX_CHARS = 2200      # total context cap
VERSE_MAX_CHARS   = 450       # per-verse cap
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


## Section 1 â€” Load Data

We load the Quran translations CSV and keep only the `sahih` translation column alongside `Surah` and `Verse` identifiers. A composite `verse_id` (e.g. `4:36`) is created for citation purposes.

In [2]:
def find_dataset(filename="quran_translations.csv"):
    candidates = [
        Path("..") / filename,
        Path(".") / filename,
        Path("../.." ) / filename,
        Path(__file__).parent.parent / filename if "__file__" in dir() else Path(".") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "Could not find " + repr(filename) + ".\n"
        "Place it one level above the Claude/ folder."
    )

CSV_PATH = find_dataset()
df = pd.read_csv(CSV_PATH)
print("Rows:", len(df), " | Columns:", list(df.columns[:6]), "...")
df = df[["Surah", "Verse", "sahih"]].dropna()
df["verse_id"] = df["Surah"].astype(str) + ":" + df["Verse"].astype(str)
print("Verses after dropna:", len(df))
df.head(3)

Rows: 6236  | Columns: ['Surah', 'Verse', 'ahmedali', 'ahmedraza', 'arberry', 'daryabadi'] ...
Verses after dropna: 6236


,Surah,Verse,sahih,verse_id
0,1,1,"In the name of Allah, the Entirely Merciful, t...",1:1
1,1,2,"[All] praise is [due] to Allah, Lord of the wo...",1:2
2,1,3,"The Entirely Merciful, the Especially Merciful,",1:3


## Section 2 â€” Preprocess

Before building the FAISS index we clean the text:

- Lowercase everything
- Remove punctuation (keep apostrophes for contractions)
- Tokenize with NLTK `word_tokenize`
- Remove English stopwords

Note: we embed the **original** `sahih` text (not the cleaned version) so that the LLM receives readable passages. The cleaned text is used only for optional keyword-based analysis.

In [3]:
PUNCT_RE = re.compile(r"[^a-z0-9\s']+", re.IGNORECASE)
STOP     = set(stopwords.words("english"))

def preprocess_text(text):
    text  = text.lower()
    text  = PUNCT_RE.sub(" ", text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP and t.strip() != ""]
    return " ".join(tokens)

df["clean"] = df["sahih"].apply(preprocess_text)
print("Verses after cleaning:", len(df))


Verses after cleaning: 6236


## Section 3 â€” Build FAISS Index

We use `all-MiniLM-L6-v2` (384-dimensional dense vectors) to encode every verse.

Key choices:
- `normalize_embeddings=True` so that inner-product search equals cosine similarity
- `IndexFlatIP` (inner product) â€” exact search, no approximation
- Vectors cast to `float32` and made contiguous for FAISS compatibility

For a production system you would use `IndexIVFFlat` or `IndexHNSW` for faster approximate search over millions of documents.

In [4]:
print("Loading sentence encoder:", MODEL_NAME)
st_model = SentenceTransformer(MODEL_NAME)

# --- Embedding cache ---
# Embeddings are saved to  <repo>/embeddings/<model>_<column>.npy
# If the file already exists we load it instead of re-encoding.
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
cache_file = EMBEDDINGS_DIR / f"{MODEL_NAME.replace('/', '_')}_{TEXT_COL}.npy"

if cache_file.exists():
    print("Loading cached embeddings from:", cache_file)
    corpus_embeddings = np.load(str(cache_file))
    print("Loaded shape:", corpus_embeddings.shape)
else:
    print("Encoding", len(df), "verses (column:", TEXT_COL, ")...")
    t0 = time.perf_counter()
    corpus_embeddings = st_model.encode(
        df[TEXT_COL].tolist(),
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)
    elapsed = time.perf_counter() - t0
    print("Encoded in", round(elapsed, 2), "s")
    np.save(str(cache_file), corpus_embeddings)
    print("Saved embeddings to:", cache_file)

corpus_embeddings = np.ascontiguousarray(corpus_embeddings)
print("Shape:", corpus_embeddings.shape)

d = corpus_embeddings.shape[1]
index_flat = faiss.IndexFlatIP(d)
index_flat.add(corpus_embeddings)
print("FAISS index size:", index_flat.ntotal)

Loading sentence encoder: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding 6236 verses (column: sahih )...


Batches:   0%|          | 0/98 [00:00<?, ?it/s]

Encoded in 43.47 s
Saved embeddings to: ..\embeddings\all-MiniLM-L6-v2_sahih.npy
Shape: (6236, 384)
FAISS index size: 6236


## Section 4 â€” Retrieve

Given a natural-language query, we:

1. Encode it with the same model (normalized)
2. Run `index.search(vec, k)` â€” returns cosine similarity scores and row indices
3. Map indices back to the DataFrame to get verse text and IDs

The result is a ranked list of the `TOP_K` most semantically similar verses.

In [5]:
def encode_query_vec(query):
    """Encode a query string to a normalized float32 FAISS-ready vector."""
    vec = st_model.encode([query], normalize_embeddings=True).astype(np.float32)
    return np.ascontiguousarray(vec)

# Inspect the query vector directly
sample_vec = encode_query_vec("mercy and forgiveness")
print("Query vector shape :", sample_vec.shape)
print("L2 norm            :", round(float(np.linalg.norm(sample_vec)), 6), " (should be 1.0)")
print("First 8 dims       :", sample_vec[0, :8].round(4))

Query vector shape : (1, 384)
L2 norm            : 1.0  (should be 1.0)
First 8 dims       : [-0.0671  0.1058  0.0045 -0.0137  0.0367  0.0477  0.0349 -0.0616]


In [7]:
def retrieve(query, k=TOP_K):
    vec = encode_query_vec(query)
    scores, indices = index_flat.search(vec, k)
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        row = df.iloc[idx]
        results.append({
            "rank": rank,
            "verse_id": row["verse_id"],
            "text": row["sahih"],
            "score": float(score),
        })
    return results

# Demo retrieval
demo_q = "parents right in islam"
hits = retrieve(demo_q)
print("Query:", demo_q)
print()
for h in hits:
    print("Rank", h["rank"], " Score:", round(h["score"], 4), " [", h["verse_id"], "]")
    print(" ", h["text"][:120])
    print()

Query: parents right in islam

Rank 1  Score: 0.5099  [ 4:33 ]
  And for all, We have made heirs to what is left by parents and relatives. And to those whom your oaths have bound [to yo

Rank 2  Score: 0.5032  [ 72:14 ]
  And among us are Muslims [in submission to Allah], and among us are the unjust. And whoever has become Muslim - those ha

Rank 3  Score: 0.4997  [ 37:149 ]
  So inquire of them, [O Muhammad], "Does your Lord have daughters while they have sons?

Rank 4  Score: 0.4965  [ 58:2 ]
  Those who pronounce thihar among you [to separate] from their wives - they are not [consequently] their mothers. Their m



## Section 5 â€” Augment (Prompt Building)

The **Augment** step turns raw retrieval results into a structured prompt:

1. Each retrieved verse is prefixed with its `[Surah:Verse]` citation tag
2. Verses longer than `VERSE_MAX_CHARS` are truncated to keep the prompt manageable
3. The total context is capped at `CONTEXT_MAX_CHARS` to avoid exceeding the model's token limit
4. A system instruction tells the model to use **only** the provided context and to cite verse IDs

This grounding instruction is critical: without it, seq2seq models often ignore the context entirely.

In [8]:
SYSTEM_PROMPT = (
    "You are a careful assistant. "
    "Use ONLY the context passages below to answer the question. "
    "Cite the verse IDs (e.g. [4:36]) when you make a claim. "
    "If the context does not contain the answer, say: "
    "'I cannot tell from the provided context.'"
)

def build_rag_prompt(query, retrieved):
    blocks = []
    for h in retrieved:
        text = h["text"]
        if len(text) > VERSE_MAX_CHARS:
            text = text[:VERSE_MAX_CHARS] + "..."
        blocks.append("[" + h["verse_id"] + "] " + text)
    context = "\n\n".join(blocks)
    if len(context) > CONTEXT_MAX_CHARS:
        context = context[:CONTEXT_MAX_CHARS] + "\n\n[... context truncated ...]"
    prompt = (
        SYSTEM_PROMPT
        + "\n\n--- Context ---\n"
        + context
        + "\n\n--- Question ---\n"
        + query
        + "\n\n--- Answer ---\n"
    )
    return prompt

demo_prompt = build_rag_prompt(demo_q, hits)
print("Prompt length:", len(demo_prompt), "chars")
print()
print(demo_prompt)

Prompt length: 1070 chars

You are a careful assistant. Use ONLY the context passages below to answer the question. Cite the verse IDs (e.g. [4:36]) when you make a claim. If the context does not contain the answer, say: 'I cannot tell from the provided context.'

--- Context ---
[4:33] And for all, We have made heirs to what is left by parents and relatives. And to those whom your oaths have bound [to you] - give them their share. Indeed Allah is ever, over all things, a Witness.

[72:14] And among us are Muslims [in submission to Allah], and among us are the unjust. And whoever has become Muslim - those have sought out the right course.

[37:149] So inquire of them, [O Muhammad], "Does your Lord have daughters while they have sons?

[58:2] Those who pronounce thihar among you [to separate] from their wives - they are not [consequently] their mothers. Their mothers are none but those who gave birth to them. And indeed, they are saying an objectionable statement and a falsehood. But in

## Section 6 â€” Generate

We use `google/flan-t5-small` as our local generator:

- It is a seq2seq model fine-tuned on instruction-following tasks (~300 MB)
- Small enough to run on CPU in a few seconds
- Accepts long input sequences (up to 512 tokens after truncation)

For production you would swap this for a larger model: GPT-4o via the OpenAI API, Claude via the Anthropic API, or a local Llama-3 / Mistral model via `llama.cpp` or `vllm`.

Set `USE_LOCAL_LLM = False` in the config cell to skip generation entirely (retrieval-only mode).

In [9]:
if USE_LOCAL_LLM:
    print("Loading generator:", GEN_MODEL, "(~300 MB first run)")
    gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
    gen_model     = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL).to(DEVICE)
    gen_model.eval()
    print("Generator ready on", DEVICE)
else:
    print("USE_LOCAL_LLM=False -- skipping generator load")
    gen_model = None
    gen_tokenizer = None

Loading generator: google/flan-t5-small (~300 MB first run)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator ready on cpu


In [10]:
def generate_answer(prompt):
    if gen_model is None:
        return "[Generation skipped -- set USE_LOCAL_LLM=True]"
    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(DEVICE)
    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            early_stopping=True,
        )
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)

rag_answer = generate_answer(demo_prompt)
print("Query :", demo_q)
print()
print("RAG Answer:")
print(rag_answer)

Query : parents right in islam

RAG Answer:
[4:33]


## Section 7 â€” RAG vs No-Context Comparison

To appreciate what retrieval buys us, we compare two conditions:

**No-context (baseline):** The model is asked the question directly. It can only draw on parametric memory â€” knowledge stored in its weights during pretraining. For a small model like flan-t5-small, this often produces vague, generic, or hallucinated answers.

**RAG (with context):** The model receives the top-k retrieved verses as grounding material. The answer should be more specific, faithful to the corpus, and citable.

The gap between the two answers illustrates exactly why RAG was proposed: it shifts the burden from "what did the model memorize?" to "what does the retrieved evidence say?"

In [11]:
# Baseline: ask the model directly without any retrieved context
baseline_prompt = "Answer the following question: " + demo_q

baseline_answer = generate_answer(baseline_prompt)
rag_answer2     = generate_answer(build_rag_prompt(demo_q, hits))

print("=== Query ===")
print(demo_q)
print()
print("=== No-Context Answer ===")
print(baseline_answer)
print()
print("=== RAG Answer (with", len(hits), "retrieved verses) ===")
print(rag_answer2)

=== Query ===
parents right in islam

=== No-Context Answer ===
islam

=== RAG Answer (with 4 retrieved verses) ===
[4:33]


In [12]:
q2 = "patience and gratitude in times of hardship"
hits2   = retrieve(q2)
prompt2 = build_rag_prompt(q2, hits2)
answer2 = generate_answer(prompt2)

print("Query:", q2)
print()
print("Retrieved verses:")
for h in hits2:
    print(" [" + h["verse_id"] + "] score=" + str(round(h["score"], 3)))
    print("  " + h["text"][:100])
print()
print("Answer:", answer2)

Query: patience and gratitude in times of hardship

Retrieved verses:
 [2:243] score=0.561
  Have you not considered those who left their homes in many thousands, fearing death? Allah said to t
 [27:40] score=0.518
  Said one who had knowledge from the Scripture, "I will bring it to you before your glance returns to
 [18:46] score=0.512
  Wealth and children are [but] adornment of the worldly life. But the enduring good deeds are better 
 [90:16] score=0.511
  Or a needy person in misery

Answer: [2:24]


## What This Notebook Is NOT

This is a teaching prototype — intentionally minimal. Before building on it, understand what it deliberately omits:

| Missing piece | Why it matters |
|---|---|
| Re-ranking step | Top-k by cosine alone can include off-topic verses; a cross-encoder re-ranker would improve precision |
| Faithfulness evaluation | We never verify the answer is actually supported by the retrieved passages |
| Grounding check | The model may answer from parametric memory and ignore the context entirely |
| Safety filters | No hallucination detection, no toxicity check, no output validation |
| Evaluation suite | No NDCG, no RAGAS, no human eval — we have no idea how often the answers are correct |
| Conversation history | Single-turn only; follow-up questions lose all context |
| Production indexing | IndexFlatIP over the full corpus is fine for a demo; millions of docs need IVF or HNSW |

Knowing what a system does NOT do is as important as knowing what it does. Each row above is a research area or engineering challenge in its own right.

## Section 8 â€” Limitations and What's Next

Our RAG prototype works end-to-end, but there are several important limitations in a production deployment:

| Limitation | Impact | Production Fix |
|---|---|---|
| No re-ranking | Top-k may include off-topic verses | Cross-encoder re-ranker |
| flan-t5-small is weak | Short, sometimes irrelevant answers | Use GPT-4o / Claude / Llama-3 |
| No faithfulness check | Answer may ignore context | NLI-based grounding verifier |
| No eval metrics | Cannot measure answer quality | RAGAS, faithfulness, answer relevance |
| Fixed k=4 | May miss coverage or add noise | Adaptive k based on score threshold |
| No chat history | Single-turn only | Conversation memory / sliding window |

### Directions for improvement

- **Hybrid retrieval:** BM25 (sparse, keyword) + SBERT (dense, semantic) fused with Reciprocal Rank Fusion
- **Re-ranking:** Use a cross-encoder (e.g. `cross-encoder/ms-marco-MiniLM-L-6-v2`) to re-score the top-20 candidates before passing top-4 to the LLM
- **Evaluation:** RAGAS framework measures faithfulness, answer relevance, and context precision automatically
- **Streaming:** For interactive apps, stream the LLM output token by token for a better UX

## Section 9 â€” Workshop Summary

This workshop has taken you on a journey from the very first steps of NLP to a working Retrieval-Augmented Generation system. Here is the full arc:

| Notebook | Topic | Key Idea |
|---|---|---|
| NB01 | Text preprocessing, BoW | Represent text as word counts |
| NB02 | TF-IDF and ranked retrieval | Weight words by rarity across docs |
| NB03 | N-grams and context | Capture local word sequences |
| NB04 | Word2Vec embeddings | Dense vectors that capture meaning |
| NB05 | FastText and subword models | Handle morphology and OOV words |
| NB06 | Sentence embeddings (SBERT) | One vector per sentence, not per word |
| NB07 | Vector databases (FAISS) | Index millions of vectors for fast ANN search |
| NB08 | RAG and LLM-based retrieval | Ground a language model in a real corpus |

Each step addressed a limitation of the previous one: bag-of-words missed word order, TF-IDF missed semantics, Word2Vec missed sentence-level meaning, FAISS made search scalable, and RAG turned search results into grounded natural-language answers.

**You have traveled from counting words to grounding a language model in a corpus â€” the full arc of modern NLP.**

## Reflection Questions

1. **Why does normalizing embeddings before inner-product search give cosine similarity?**
   Cosine similarity between two vectors u and v is defined as (u . v) / (||u|| * ||v||). When both vectors are L2-normalized (||u|| = ||v|| = 1), the denominator equals 1, so the dot product equals the cosine similarity directly. FAISS IndexFlatIP computes exact inner products, so normalizing first turns it into exact cosine search.

2. **What would happen if you retrieved 20 verses but the model only sees 512 tokens?**
   The tokenizer would truncate the input at 512 tokens. Verses near the end of the context window would be silently cut off, potentially removing the most relevant passages (or leaving only partial passages). This can cause the model to miss key information or produce incoherent answers. Solutions: rank verses by importance and put the best ones first, or use a model with a longer context window.

3. **flan-t5-small sometimes ignores the context. Why might that happen?**
   flan-t5-small was fine-tuned on a diverse mixture of tasks with short answers. It may have learned shortcuts that produce fluent-sounding answers from parametric memory without reading the provided context carefully. The grounding instruction in the system prompt helps, but a small model has limited capacity to follow complex instructions. Larger models (flan-t5-xl, flan-ul2, Llama-3) are much better at respecting context constraints.

4. **How would you evaluate whether the RAG answer is actually grounded in the retrieved verses?**
   Use an NLI (Natural Language Inference) model to check whether each sentence in the answer is entailed by at least one retrieved verse. The RAGAS framework automates this with a "faithfulness" score: it decomposes the answer into atomic claims and asks an LLM to verify each claim against the context. A faithfulness score of 1.0 means every claim is supported; 0.0 means none are.

5. **Design a 2-stage retrieval system using BM25 (sparse) + SBERT (dense). What are the trade-offs?**
   Stage 1 (Recall): Use BM25 to retrieve the top-100 candidates using keyword overlap. BM25 is fast, interpretable, and excellent at exact-match queries. Stage 2 (Precision): Re-rank the 100 candidates with SBERT cosine similarity and keep the top-4. SBERT captures semantic meaning that BM25 misses. Trade-offs: the combined system is slower and more complex than either alone, but achieves higher recall than dense-only (BM25 catches exact keywords) and higher precision than sparse-only (SBERT handles paraphrases and synonyms). This hybrid approach is used in production systems like Elasticsearch with vector fields.